In [8]:
#julia version:1.11.6
include("CRD_STA.jl")
using Plots
using LinearAlgebra
using NonlinearEigenproblems
using DelimitedFiles

In [ ]:
N_cheb = 99
Mr = 0.1
gamma = 1.4
sigma = 0.72
Ro = -1
Co = 2-Ro-Ro^2
Tw = 1
R = 285.36
Ma = Mr/R
omega = 0
be1 = 0.07759
c = 0.5
u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,"cheb");
H,T = T_ca(Mr,f,q,w0,gamma,Tw);
F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim");
# lam = - (2/3) * T;
# kappa = (1/sigma) * T;
# A0,A1,A2 = Spatial_mode(-F,-G,-H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be1,N_cheb)
# nep = PEP([A0,A1,A2]); 
# eigval,eigvec = iar(nep,σ = 0.38 , neigs = 1 ,maxit = 500,tol=1e-10)
# eigval

In [ ]:
range(-0.,0.15,length = 10)

-0.3:0.05:0.15

In [77]:
function BF(N_cheb,Ro,Tw,Mr)
    gamma = 1.4
    sigma = 0.72
    Co = 2 - Ro - Ro^2
    u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co)
    H,T = T_ca(Mr,f,q,w0,gamma,Tw)
    F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
    lam = - (2/3) * T
    kappa = (1/sigma) * T
    return F,G,H,T,rho,z,lam,kappa,D,D2
end
function eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num)
    sigma = 0.72
    gamma = 1.4
    A0,A1,A2 = Spatial_mode_BEK(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2)
    nep = PEP([A0,A1,A2]); 
    eigval,eigvec = iar(nep,σ = c , neigs = num ,maxit = 500,tol=1e-10)
    eigval = conj(eigval)
    return eigval
end

eigsol (generic function with 2 methods)

In [101]:
##DIRECTLY CACULATE CUR
##initial
# for omega = [0.048,0.024,-0.024,-0.048]
function cur(Tw,Mr,Ro,omega,R_ini,c_ini,be_ini)

    N_cheb = 69
    gamma = 1.4
    sigma = 0.72
    R_step = 0.4
    be_step = -0.0008
    num = 1
    Co = 2-Ro-Ro^2
    Ma = Mr/R_ini
    F,G,H,T,rho,z,lam,kappa,D,D2 = BF(N_cheb,Ro,Tw,Mr)
    initial = []
    tempvec_1 = [0 0 0 0 0]
    tempvec_2 = [0 0 0 0 0]
    eigval = 0
    writedlm("output.dat",initial)
    writedlm("output_eig.dat",initial)
    eigval = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R_ini,Ma,omega,be_ini,N_cheb,Ro,Co,D,D2,c_ini,num)
    if imag(eigval[1]) < 0
        for be = be_ini : be_step : -0.5
            sig_last = sign(imag(eigval[1]))
            eigval = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R_ini,Ma,omega,be,N_cheb,Ro,Co,D,D2,c_ini,num)
            sig_now = sign(imag(eigval[1]))
            # point = filter(x -> abs(imag(x)) < 0.0004 , eigval)
            open("output_eig.dat", "a") do io
                write(io,"be=$be,eig=$eigval\n")
            end
            if sig_last * sig_now < 0 && sig_last != 0 
                initial = [omega R_ini be real(eigval[1]) imag(eigval[1])]
                break
            end
        end
    elseif imag(eigval[1]) > 0
        for be = be_ini : -be_step : 0.5
            sig_last = sign(imag(eigval[1]))
            eigval = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R_ini,Ma,omega,be,N_cheb,Ro,Co,D,D2,c_ini,num)
            sig_now = sign(imag(eigval[1]))
            open("output_eig.dat", "a") do io
                write(io,"be=$be,eig=$eigval\n")
            end
            if sig_last * sig_now < 0 && sig_last != 0
                initial = [omega R_ini be real(eigval[1]) imag(eigval[1])]
                break
            end
        end
    end
    total = initial

 # CACULATE

    for be = initial[end,3] +  be_step  :  -1 * be_step : 0.4

        if total[end,2] < 80
            num = 1
        else
            num = 1
        end

        c = total[end,4] - im * total[end,5]
        eigval_1 = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,total[end,2],Ma,omega,be+2*be_step,N_cheb,Ro,Co,D,D2,c,num)
        eigval_2 = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,total[end,2],Ma,omega,be-2*be_step,N_cheb,Ro,Co,D,D2,c,num)
        index1 = findmin(x-> abs.(imag(x)) , eigval_1)[2]
        index2 = findmin(x-> abs.(imag(x)) , eigval_2)[2]
        if (imag(eigval_1[index1]) < 0 && imag(eigval_2[index2]) > 0) || (imag(eigval_1[index1]) < 0 && imag(eigval_2[index2]) < 0)

            mode = 1

        elseif (imag(eigval_1[index1]) > 0 && imag(eigval_2[index2]) < 0) || (imag(eigval_1[index1]) > 0 && imag(eigval_2[index2]) > 0)

            mode = 2

        end
        
        if mode == 1 



            for R = total[end,2] : R_step : 600

                Ma = Mr/R
                if abs(total[end,3] - be) > -2 * be_step
                    c = total[end,4] + 0.3 - im * total[end,5]
                else
                    c = total[end,4] - im * total[end,5]
                end
                eigval = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num)
                index = findmin(abs.(imag(eigval)))

                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1])]]
                tempvec_2 = [tempvec_2;[omega R be real(eigval[end]) imag(eigval[end])]]
                len = size(tempvec_1,1)

                open("output.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end

                if tempvec_1[end-1, 5] * tempvec_1[end,5] < 0 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]

                    break
                elseif tempvec_2[end-1, 5] * tempvec_2[end, 5]<0

                    total = [total ; tempvec_2[end:end,:]]                    
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]

                    break
                end
                
                if (len > 5 && abs(tempvec_1[end,5]) > abs(tempvec_1[end-2,5])) || len > 100

                    mode = 2
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]
                    break
                end
            end        
        end


        if mode == 2

            for R = total[end,2]: -R_step : 0

                Ma = Mr/R
                if abs(total[end,3] - be) > -2 * be_step
                    c = total[end,4] + 0.3 - im * total[end,5]
                else
                    c = total[end,4] - im * total[end,5]
                end
                
                eigval = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num)
                index = findmin(abs.(imag(eigval)))
                
                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1])]]
                tempvec_2 = [tempvec_2;[omega R be real(eigval[end]) imag(eigval[end])]]
                len = size(tempvec_1,1)

                open("output.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end

                if tempvec_1[end-1, 5] * tempvec_1[end,5] < 0 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]

                    break
                elseif tempvec_2[end-1, 5] * tempvec_2[end, 5]<0

                    total = [total ; tempvec_2[end:end,:]]                    
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]

                    break
                end
                
                if (len > 5 && abs(tempvec_1[end,5]) > abs(tempvec_1[end-2,5])) || len > 100

                    mode = 1
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]
                    break
                end
            end        
        end

        if mode == 1
            
            for R = total[end,2]: R_step : 600
                Ma = Mr/R

                if total[end,3] == be

                    break

                end 

                if abs(total[end,3] - be) > -2 * be_step
                    c = total[end,4] + 0.3 - im * total[end,5]
                else
                    c = total[end,4] - im * total[end,5]
                end
                
                eigval = eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num)
                index = findmin(abs.(imag(eigval)))
                
                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1])]]
                tempvec_2 = [tempvec_2;[omega R be real(eigval[end]) imag(eigval[end])]]
                len = size(tempvec_1,1)

                open("output.dat", "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len\n")
                end

                if tempvec_1[end-1, 5] * tempvec_1[end,5] < 0 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]

                    break
                elseif tempvec_2[end-1, 5] * tempvec_2[end, 5]<0

                    total = [total ; tempvec_2[end:end,:]]                    
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]

                    break
                end
                
                if (len > 5 && abs(tempvec_1[end,5]) > abs(tempvec_1[end-2,5])) || len > 100

                    mode = 2
                    tempvec_1 = [0 0 0 0 0]
                    tempvec_2 = [0 0 0 0 0]
                    break
                end
            end        
        end

        if total[end,2] > R_ini + 10 && size(total,1)>40
            break
        end
    end
    dir = "omega=$(omega)"
    mkpath(dir)
    filename = "ome=$(omega)_Tw=$(Tw)_Mr=$(Mr).dat"
    path = joinpath(dir,filename)
    str1 = "Variables=\"omega\" \"R\" \"beta\" \"alpha_i\" \"alpha_r\""
    str2 = "Zone T=\"omega=$(omega),Ma=$(Mr),Tw=$(Tw),Ro=$(Ro)\""
    open(path,"w") do io
        println(io,str1)
        println(io,str2)
        writedlm(io,total[2:end,:])
    end
end

cur (generic function with 2 methods)

In [102]:
for omega in -0.05
    for Tw in 0.8 : 0.1 : 1.2
        for Mr in 0.3 : 0.3 : 1.2
            c_ini = 0.03
            be_ini = omega + 0.02
            R_ini = 200
            Ro = 0.687
            cur(Tw,Mr,Ro,omega,R_ini,c_ini,be_ini)
        end
    end
end

InterruptException: InterruptException: